# Tarea 26 · Modelo de clasificación: Regresión Logística

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/26_regresion_logistica.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**
**Integrantes:** Matías, Julián y Emmanuel.

Entrenamos un modelo que predice si un cliente se va (`Churn = 1`) o se queda (`Churn = 0`), usando los datos de entrenamiento que dejó el preprocesamiento (tareas 21 a 24). La **evaluación** del modelo (matriz de confusión y accuracy sobre test) corresponde a la tarea 27, y la interpretación de sus coeficientes, a la tarea 28.

**Conceptos de esta tarea**

- **Clasificación:** predecir una categoría (aquí, "se va" o "se queda") en lugar de un número.
- **Aprendizaje supervisado:** el modelo aprende de ejemplos que ya tienen la respuesta correcta (`y_train`).
- **Entrenar (`fit`):** que el modelo ajuste sus parámetros usando solo los datos de entrenamiento.

## Cómo ejecutarlo

- **Local:** instalá las dependencias de `requirements.txt`, abrí el notebook desde el repositorio y ejecutá todas las celdas.
- **Colab:** abrí el botón y elegí **Entorno de ejecución → Ejecutar todo**. Los datos se leen desde `main` de GitHub.

## 1. Carga de los datos de entrenamiento

Leemos **solo** `X_train` e `y_train`. En esta tarea no se leen los archivos de test: así se garantiza que el modelo no vea esos datos mientras se entrena (prevención de *data leakage*, que la rúbrica evalúa). `fila_origen` es el número de fila del cliente en el dataset original; se usa como índice para alinear cada cliente con su etiqueta y **nunca como variable predictora**.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

GITHUB_REPO = 'julianschmidt1/tpi-analisis-de-datos'
GITHUB_REF = 'main'
RUTA_X = Path('data/processed/X_train.csv')
RUTA_Y = Path('data/processed/y_train.csv')

if EN_COLAB:
    RAIZ = Path.cwd()
    base = f'https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_REF}/'
    fuente_x, fuente_y = base + RUTA_X.as_posix(), base + RUTA_Y.as_posix()
else:
    actual = Path.cwd().resolve()
    RAIZ = next(
        (ruta for ruta in (actual, *actual.parents) if (ruta / RUTA_X).is_file()),
        None,
    )
    if RAIZ is None:
        raise FileNotFoundError('No se encontró X_train.csv. Abrí Jupyter dentro del repositorio.')
    fuente_x, fuente_y = RAIZ / RUTA_X, RAIZ / RUTA_Y

X_train = pd.read_csv(fuente_x, index_col='fila_origen')
y_train = pd.read_csv(fuente_y, index_col='fila_origen')['Churn']

assert X_train.index.equals(y_train.index), 'Cada cliente debe coincidir con su etiqueta.'
assert not X_train.isna().any().any(), 'No debe haber valores nulos.'
assert set(y_train.unique()) == {0, 1}, 'El objetivo debe ser 0 (se queda) o 1 (se va).'

print(f'Clientes de entrenamiento: {len(X_train):,}')
print(f'Variables predictoras: {X_train.shape[1]}')
print(f'Bajas en entrenamiento (Churn = 1): {y_train.mean():.1%}')

Clientes de entrenamiento: 5,634
Variables predictoras: 46
Bajas en entrenamiento (Churn = 1): 26.5%


## 2. Qué modelo usamos y por qué

Entrenamos **un único modelo de clasificación elemental**. Elegimos **Regresión Logística** por cuatro motivos:

1. **El objetivo es binario** (se va / se queda), que es exactamente el tipo de problema para el que sirve.
2. **Devuelve una probabilidad**, no solo una etiqueta. Por ejemplo, "este cliente tiene 34 % de probabilidad de irse". El modelo pasa a "se va" cuando esa probabilidad supera 0,5.
3. **Es interpretable:** cada variable recibe un coeficiente que indica si empuja hacia la baja o hacia la permanencia. Eso permite sacar conclusiones de negocio (tarea 28).
4. **Es sensible a la escala de las variables.** Por eso en la tarea 23 se estandarizaron las variables numéricas: así ninguna domina el ajuste solo por tener valores más grandes.

**Parámetros elegidos**

- `max_iter=1000`: la cantidad máxima de pasos de ajuste. El valor por defecto (100) puede quedarse corto; verificamos más abajo que el modelo realmente converge.
- `random_state=42`: fija la semilla para que los resultados se repitan idénticos.
- El resto queda por defecto, entre ello una **regularización** (`C=1,0`) que evita que los coeficientes crezcan demasiado. Esto ayuda en este dataset, donde varias variables están relacionadas entre sí (por ejemplo, las columnas de facturación calculadas).

## 3. Entrenamiento

`fit(X_train, y_train)` es el paso en que el modelo aprende. Se hace **únicamente** con los datos de entrenamiento.

In [2]:
modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)

iteraciones = int(modelo.n_iter_[0])
print(f'Modelo entrenado con {len(X_train):,} clientes y {X_train.shape[1]} variables.')
print(f'Iteraciones usadas: {iteraciones} de {modelo.max_iter} (converge: {iteraciones < modelo.max_iter}).')
print(f'Ordenada al origen (intercepto): {modelo.intercept_[0]:.3f}')
print(f'Cantidad de coeficientes: {modelo.coef_.shape[1]}')
assert iteraciones < modelo.max_iter, 'El modelo no convergió: aumentar max_iter.'

Modelo entrenado con 5,634 clientes y 46 variables.
Iteraciones usadas: 40 de 1000 (converge: True).
Ordenada al origen (intercepto): -0.370
Cantidad de coeficientes: 46


## 4. Comprobación rápida del entrenamiento

Antes de pasar a la evaluación, verificamos que el modelo aprendió algo con sentido. Comparamos su **accuracy sobre los propios datos de entrenamiento** con la **línea base**: la accuracy que tendría un "modelo" que siempre responde la clase más frecuente (nadie se va).

Esta comprobación no es la evaluación del modelo. Un modelo siempre rinde mejor sobre los datos con los que se entrenó. La medición honesta se hace en la tarea 27, sobre los datos de test, que acá no se tocaron.

In [3]:
linea_base = 1 - y_train.mean()
acc_entrenamiento = modelo.score(X_train, y_train)
proba_baja = modelo.predict_proba(X_train)[:, 1]
prediccion = modelo.predict(X_train)

print(f'Línea base (siempre "se queda"): {linea_base:.1%}')
print(f'Accuracy sobre entrenamiento:     {acc_entrenamiento:.1%}')
print(f'Bajas predichas: {prediccion.mean():.1%} (reales en entrenamiento: {y_train.mean():.1%})')
print()
print('Probabilidad de baja y predicción para los primeros 5 clientes:')
ejemplo = pd.DataFrame({
    'prob_baja': proba_baja[:5].round(3),
    'prediccion': prediccion[:5],
    'real': y_train.to_numpy()[:5],
}, index=X_train.index[:5])
ejemplo

Línea base (siempre "se queda"): 73.5%
Accuracy sobre entrenamiento:     80.6%
Bajas predichas: 22.1% (reales en entrenamiento: 26.5%)

Probabilidad de baja y predicción para los primeros 5 clientes:


,prob_baja,prediccion,real
fila_origen,,,
3738,0.338,0,0
3151,0.291,0,0
4860,0.044,0,0
3867,0.090,0,0
3810,0.378,0,0


### Interpretación

- El modelo **converge** y mejora la línea base sobre entrenamiento (80,6 % frente a 73,5 %). Aprendió una relación real entre las variables y la baja.
- Predice menos bajas de las que hay realmente (22,1 % frente a 26,5 %): es un modelo algo conservador. Con el umbral de 0,5, deja sin detectar a algunos clientes que sí se van. Cuánto cuesta ese error se analiza con la matriz de confusión en la tarea 27.
- La mejora sobre la línea base no alcanza como conclusión. Hay que confirmarla **sobre test**: ahí se ve si el modelo generaliza o si solo memorizó el entrenamiento.

## 5. Coeficientes aprendidos

Guardamos los coeficientes para la tarea 28 (interpretación). Para cada variable se calcula además el **odds ratio** (`exp(coeficiente)`): un valor mayor que 1 indica que, con todo lo demás igual, la variable **aumenta** las chances de baja, y uno menor que 1 que las **reduce**. Las variables numéricas están estandarizadas, así que su coeficiente se lee "por cada desvío estándar de aumento".

In [4]:
coeficientes = pd.DataFrame({
    'variable': X_train.columns,
    'coeficiente': modelo.coef_[0],
})
coeficientes['odds_ratio'] = np.exp(coeficientes['coeficiente'])
coeficientes = coeficientes.reindex(
    coeficientes['coeficiente'].abs().sort_values(ascending=False).index
).reset_index(drop=True)

RUTA_SALIDA = RAIZ / 'data/processed/26_coeficientes_logistica.csv'
coeficientes.to_csv(RUTA_SALIDA, index=False, encoding='utf-8')
print(f'Archivo generado: {RUTA_SALIDA.relative_to(RAIZ).as_posix()}')
coeficientes.head(10).round(3)

Archivo generado: data/processed/26_coeficientes_logistica.csv


,variable,coeficiente,odds_ratio
0,numericas__tenure,-1.243,0.289
1,contrato__Contract,-0.689,0.502
2,nominal__InternetService_Fiber optic,0.667,1.948
3,nominal__InternetService_DSL,-0.616,0.540
4,numericas__MonthlyCharges,-0.586,0.557
5,nominal__PaperlessBilling_No,-0.294,0.745
6,nominal__OnlineBackup_No internet service,-0.266,0.766
7,nominal__OnlineSecurity_No internet service,-0.266,0.766
8,nominal__InternetService_No,-0.266,0.766
9,nominal__TechSupport_No internet service,-0.266,0.766


**Primera lectura:**

- Las variables con mayor peso son `tenure` (la antigüedad), el tipo de contrato y el servicio de internet. Coincide con lo que mostraron los gráficos de la tarea 17: los clientes nuevos son los que más se van.
- Algunos coeficientes pueden parecer contraintuitivos (por ejemplo, `MonthlyCharges` con signo negativo aunque quienes se van pagan más). Pasa porque varias variables miden cosas parecidas (cargo mensual, total e ingreso estimado) y el modelo reparte el efecto entre ellas. **Los coeficientes individuales deben interpretarse con cuidado**, y esa discusión se hace en la tarea 28.

## 6. Cómo usar este modelo en las tareas 27 y 28

El modelo **no se guarda en un archivo**: se reentrena en segundos y de forma idéntica, ya que los datos y la semilla son fijos. Para evaluarlo sobre test (tarea 27), copiar esta celda:

```python
import pandas as pd
from sklearn.linear_model import LogisticRegression

X_train = pd.read_csv('data/processed/X_train.csv', index_col='fila_origen')
y_train = pd.read_csv('data/processed/y_train.csv', index_col='fila_origen')['Churn']
X_test = pd.read_csv('data/processed/X_test.csv', index_col='fila_origen')
y_test = pd.read_csv('data/processed/y_test.csv', index_col='fila_origen')['Churn']

modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)          # solo con entrenamiento
y_pred = modelo.predict(X_test)       # test se usa únicamente para predecir y evaluar
```

Para la tarea 28, los coeficientes ya están en `data/processed/26_coeficientes_logistica.csv`.

## 7. Resultado de la tarea

Se entrenó una **Regresión Logística** con 5.634 clientes y 46 variables, usando solo los datos de entrenamiento, con `random_state=42` y convergencia verificada. El modelo supera la línea base sobre entrenamiento (80,6 % frente a 73,5 %), aunque esa cifra no es la evaluación final. Los coeficientes quedaron exportados para la tarea 28 y el modelo quedó listo para evaluarse sobre test en la tarea 27.